In [13]:
# Télécharger les fichiers depuis GitHub
!wget https://raw.githubusercontent.com/AndryRAB/costa-verde/main/ressources/reservations_train.csv
!wget https://raw.githubusercontent.com/AndryRAB/costa-verde/main/ressources/reservations_test.csv
!wget https://raw.githubusercontent.com/AndryRAB/costa-verde/main/ressources/sample_submission.csv
!wget https://raw.githubusercontent.com/AndryRAB/costa-verde/main/ressources/data_dictionary.csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import xgboost as xgb

# 1. Chargement des données
train = pd.read_csv('reservations_train.csv')
test = pd.read_csv('reservations_test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nColonnes train:", train.columns.tolist())
print("\nAperçu train:")
print(train.head())

# 2. Analyse de la cible
print("\nDistribution de la cible:")
print(train['reservation_annulee'].value_counts(normalize=True))

# 3. Prétraitement
# Conversion des dates
for df in [train, test]:
    df['date_reservation'] = pd.to_datetime(df['date_reservation'])
    df['date_arrivee'] = pd.to_datetime(df['date_arrivee'])
    df['mois_reservation'] = df['date_reservation'].dt.month
    df['mois_arrivee'] = df['date_arrivee'].dt.month
    df['jour_semaine_arrivee'] = df['date_arrivee'].dt.dayofweek
    df['delai_jours'] = (df['date_arrivee'] - df['date_reservation']).dt.days

# Suppression des colonnes non pertinentes
cols_to_drop = ['reservation_id', 'date_reservation', 'date_arrivee']
X = train.drop(columns=cols_to_drop + ['reservation_annulee'])
y = train['reservation_annulee']
X_test = test.drop(columns=cols_to_drop)

# Identification des colonnes catégorielles et numériques
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print("\nColonnes catégorielles:", cat_cols)
print("Colonnes numériques:", num_cols)

# Pipeline de prétraitement
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# 4. Split temporel (important pour l'évaluation temporelle)
train_sorted = train.sort_values('date_reservation')
split_idx = int(len(train_sorted) * 0.8)
train_set = train_sorted.iloc[:split_idx]
val_set = train_sorted.iloc[split_idx:]

X_train = train_set.drop(columns=cols_to_drop + ['reservation_annulee'])
y_train = train_set['reservation_annulee']
X_val = val_set.drop(columns=cols_to_drop + ['reservation_annulee'])
y_val = val_set['reservation_annulee']

# 5. Modèles
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred_proba)
    acc = accuracy_score(y_val, (y_pred_proba > 0.5).astype(int))
    results[name] = {'AUC': auc, 'Accuracy': acc}
    print(f"\n{name}:")
    print(f"  AUC-ROC: {auc:.4f}")
    print(f"  Accuracy: {acc:.4f}")

# 6. Meilleur modèle
best_model_name = max(results, key=lambda x: results[x]['AUC'])
print(f"\nMeilleur modèle: {best_model_name}")

best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', models[best_model_name])
])
best_pipeline.fit(X, y)

# 7. Prédiction sur le test
y_test_proba = best_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba > 0.5).astype(int)

# 8. Création de la soumission
submission = pd.DataFrame({
    'reservation_id': test['reservation_id'],
    'probabilite_annulation': y_test_proba,
    'reservation_annulee': y_test_pred
})
submission.to_csv('submission.csv', index=False)
print("\nSoumission créée: submission.csv")
print(submission.head())

--2026-09-28 10:43:53--  https://raw.githubusercontent.com/AndryRAB/costa-verde/main/ressources/reservations_train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1471613 (1.4M) [text/plain]
Saving to: ‘reservations_train.csv.2’

reservations_train. 100%[===================>]   1.40M  --.-KB/s    in 0.05s   

2026-09-28 10:43:54 (26.4 MB/s) - ‘reservations_train.csv.2’ saved [1471613/1471613]

--2026-09-28 10:43:54--  https://raw.githubusercontent.com/AndryRAB/costa-verde/main/ressources/reservations_test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, await

### 1. Monter Google Drive

Exécutez la cellule suivante pour monter votre Google Drive. Une boîte de dialogue s'ouvrira, vous demandant de vous connecter à votre compte Google et d'autoriser Google Drive File Stream.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Trouver le chemin d'accès à vos fichiers

Après avoir monté votre Drive, vos fichiers seront accessibles via `/content/drive/MyDrive/`. Vous pouvez parcourir vos dossiers dans le panneau de gauche de Colab (l'icône de dossier) pour trouver le chemin exact de vos fichiers CSV.

Par exemple, si vos fichiers `reservations_train.csv`, `reservations_test.csv` et `sample_submission.csv` se trouvent dans un dossier nommé `mon_projet` dans votre Drive, les chemins seront :

*   `/content/drive/MyDrive/mon_projet/reservations_train.csv`
*   `/content/drive/MyDrive/mon_projet/reservations_test.csv`
*   `/content/drive/MyDrive/mon_projet/sample_submission.csv`

### 3. Mettre à jour les chemins dans votre code

Une fois que vous avez identifié les chemins corrects, vous devrez mettre à jour les lignes de code suivantes dans votre cellule `azViArkPhxvB` :

In [15]:
# Exemple de mise à jour des chemins:
# Remplacez 'votre_dossier' par le nom réel de votre dossier dans Google Drive

# train = pd.read_csv('/content/drive/MyDrive/votre_dossier/reservations_train.csv')
# test = pd.read_csv('/content/drive/MyDrive/votre_dossier/reservations_test.csv')
# sample_sub = pd.read_csv('/content/drive/MyDrive/votre_dossier/sample_submission.csv')

# Après avoir monté votre Drive, exécutez la ligne ci-dessous pour lister le contenu de votre Drive
# et vous aider à trouver le chemin de vos fichiers.
# !ls /content/drive/MyDrive

Veuillez mettre à jour les chemins dans votre code existant (`cell_id: azViArkPhxvB`) après avoir monté votre Drive.